# Bring your own data

Walk a small toy corpus through the canonical pipeline that produces the bundled `eval_deseasonalized.npy`. This notebook is self-contained: it synthesizes two fake tickers, writes them to disk as 1-minute CSVs, then runs every step from raw CSV to benchmark-ready `.npy`.

Replace the synthetic-CSV generation cell with reads of your own data and the rest of the notebook is unchanged.

In [ ]:
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data_pipeline import (
    BenchmarkReference,
    clean_ticker,
    deseasonalize,
    fit_fff,
    make_windows,
    save_benchmark_corpus,
)

## 1. Stage some raw CSVs

Each CSV is one ticker. The pipeline only requires two columns: `timestamp` (UTC ISO8601) and `close`. Anything else is ignored. We synthesize 15 trading days of minute bars (~5800 rows / ticker) with a geometric random walk on top of a U-shaped intraday volatility pattern.

In [ ]:
RAW_DIR    = Path(tempfile.mkdtemp(prefix="byod_raw_"))
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="byod_out_"))
print(f"raw CSVs   : {RAW_DIR}")
print(f"output dir : {OUTPUT_DIR}")

def synth_ticker_csv(path: Path, n_days: int = 15, seed: int = 0) -> None:
    rng = np.random.default_rng(seed)
    rows = []
    start = pd.Timestamp("2020-01-02 14:30", tz="UTC")        # 9:30 NY in Jan
    minute_of_day = np.arange(390)                            # 0..389 → 9:30..15:59
    # U-shape: high vol at open/close, calm midday
    u_shape = 1.0 + 0.6 * (np.abs(minute_of_day - 195) / 195) ** 1.5
    price = 100.0
    for d in range(n_days):
        day_open = start + pd.Timedelta(days=d)
        # skip weekends
        if day_open.dayofweek >= 5:
            continue
        ret = rng.normal(0, 1e-4, size=390) * u_shape
        closes = price * np.exp(np.cumsum(ret))
        ts = day_open + pd.to_timedelta(minute_of_day, unit="m")
        for t, c in zip(ts, closes):
            rows.append((t.isoformat(), float(c)))
        price = closes[-1]
    pd.DataFrame(rows, columns=["timestamp", "close"]).to_csv(path, index=False)

synth_ticker_csv(RAW_DIR / "AAA.csv", seed=11)
synth_ticker_csv(RAW_DIR / "BBB.csv", seed=22)
list(RAW_DIR.iterdir())

## 2. Steps 1–6 — session filter, gap handling, log returns

`clean_ticker` keeps only NYSE regular-hours bars (9:30 NY – 15:59 NY), drops any bar that follows a within-day gap > 1 minute, drops the first bar of each session (overnight), and computes 1-minute log returns. Output columns: `date_ny`, `minute_of_day_ny`, `log_return`.

In [ ]:
cleaned = {}
for csv in sorted(RAW_DIR.glob("*.csv")):
    df = clean_ticker(csv)
    cleaned[csv.stem] = df
    print(f"{csv.stem}  rows={len(df):>5}  unique_days={df['date_ny'].nunique()}")
cleaned['AAA'].head()

## 3. Step 9 — fit the intraday seasonality curve

`fit_fff` pools the cleaned returns by minute-of-day τ ∈ [0, 389], computes mean |return| per τ, and fits a 3-harmonic Fourier series. The output is a length-390 vector `s(τ)` giving the expected absolute-return scale at each minute of the session.

In production the curve is fitted **only on training tickers** (no leakage). For this self-contained demo we pool both fake tickers.

In [ ]:
s = fit_fff(list(cleaned.values()))
print(f"FFF curve  : {s.shape}  range=[{s.min():.6f}, {s.max():.6f}]")
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.arange(390), s, lw=1.5)
ax.set_xlabel("minute of session  (0 = 9:30 NY,  389 = 15:59 NY)")
ax.set_ylabel("s(τ) — expected |return|")
ax.set_title("Fitted Flexible Fourier Form")
ax.grid(alpha=0.3)

## 4. Deseasonalize and window

Divide each return by its minute-of-day factor, then slice the per-ticker series into non-overlapping windows of `WINDOW_LEN = 2520` minutes (10 trading days). With ~15 trading days per fake ticker we get one window per ticker. `regime_labels=True` tags each window 0/1 based on whether its first bar predates `CRASH_CUTOFF = 2020-02-19`.

In [ ]:
WINDOW_LEN = 2520
deseasoned = {t: deseasonalize(df, s) for t, df in cleaned.items()}
eval_arr, ticker_labels, regime_labels = make_windows(
    deseasoned, col="return_deseas",
    window_len=WINDOW_LEN, regime_labels=True,
)
print(f"eval_arr       : {eval_arr.shape}  dtype={eval_arr.dtype}")
print(f"ticker_labels  : {ticker_labels.shape}  unique={set(ticker_labels)}")
print(f"regime_labels  : {regime_labels.shape}  values={set(regime_labels.tolist())}")

## 5. Save the canonical corpus + manifest

`save_benchmark_corpus` subsamples to `BENCHMARK_N_PATHS = 200` paths, zero-means, rescales pooled std to match the reference, and writes `(N, T, 1)` float32 — the exact shape and dtype `bench.py` consumes. For this two-ticker toy corpus we only have two windows, so we skip the subsample-to-200 step and write the eval array directly.

Then write `benchmark_manifest.json` alongside it so `bench.py` knows `T`, `N`, the reference mean/std, and the seed.

In [ ]:
(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
np.save(OUTPUT_DIR / "eval_deseasonalized.npy",  eval_arr.astype(np.float32))
np.save(OUTPUT_DIR / "eval_ticker_labels.npy",   ticker_labels)
np.save(OUTPUT_DIR / "eval_regime_labels.npy",   regime_labels)
np.save(OUTPUT_DIR / "fff_pattern.npy",          s)

ref = BenchmarkReference.from_eval(OUTPUT_DIR / "eval_deseasonalized.npy")
ref.write_manifest(OUTPUT_DIR)

for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name:<28}  {p.stat().st_size:>8} bytes")
print(f"\nmanifest reference: window_len={ref.window_len}  ref_std={ref.std:.4f}  N={ref.n_paths}")

## 6. Point bench.py at the new corpus

Two ways:

1. Set `EVALFRAMEWORK_DATA_DIR` to `OUTPUT_DIR` — `evaluation_framework.paths.output_dir()` reads the env var first.
2. Copy / move the files into the repo's `data/output_data/` directory.

Once any synthetic generator's output sits next to `eval_deseasonalized.npy` as `<gen>_synthetic.npy`, add it to `generator_paths()` in `evaluation_framework/paths.py` and `bench.py` will pick it up. The toy corpus here has only two paths and won't satisfy the matched-N protocol's `N=200` default — it's enough to verify the data layout but not to score a real generator.

In [ ]:
import os
os.environ["EVALFRAMEWORK_DATA_DIR"] = str(OUTPUT_DIR)

from importlib import reload
from evaluation_framework import paths, io
reload(paths); reload(io)

real, tickers, regimes = io.load_eval_corpus()
print(f"real corpus loaded from {paths.output_dir()}")
print(f"  shape         : {real.shape}")
print(f"  tickers       : {set(tickers)}")
print(f"  regimes seen  : {set(regimes.tolist())}")

## Next steps

- **Production scale.** Run `python -m data_pipeline.data_prep <raw_csv_dir> <output_dir>` on a directory of real per-ticker CSVs to get the full ten-step pipeline (liquidity filtering, train / eval split, FFF fit on train only, optional per-ticker z-score for the training corpus).
- **Plug in synthetic outputs.** See `README.md` § *Bring your own data → synthetic outputs* for `save_benchmark_corpus`.
- **Run the benchmark.** `uv run python bench.py` once `data/output_data/` has the real corpus + at least one `<name>_synthetic.npy`.